# Consolidated evaluation of developmental prediction techniques

**Viva-ready, aggregate-only research summary**

This notebook consolidates the evidence already produced by the research codebase. It does **not** load participant-level data, fit models, select new thresholds, or create new validation evidence. The three outcomes are questionnaire-defined three-month concerns in Communication, Cognitive development, and Social-emotional development; they are not diagnoses.

## How to interpret this notebook

The first five algorithms formed the original controlled comparison. CatBoost and EBM were added in a later shared-fold comparison. LightGBM, FIGS, Histogram Gradient Boosting, and TabICLv2 were evaluated in separate development-only protocols. Their numerical results are shown together for presentation, but this notebook does not imply that all eleven were evaluated simultaneously in a single selection experiment.

Average Precision (PR-AUC) is the primary ranking metric because the positive classes are uncommon. ROC-AUC, sensitivity, specificity, balanced accuracy, and Brier score provide complementary discrimination, operating-point, and calibration evidence.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

pd.set_option('display.max_columns', 30)
pd.set_option('display.precision', 3)

ROOT = Path.cwd().resolve()
if not (ROOT / 'reports' / 'generated').is_dir():
    ROOT = ROOT.parent
REPORTS = ROOT / 'reports' / 'generated'
assert REPORTS.is_dir(), 'Run this notebook from the repository root or notebooks directory.'

def load_report(name):
    with (REPORTS / name).open(encoding='utf-8') as handle:
        return json.load(handle)

print(f'Repository: {ROOT}')
print('Privacy boundary: aggregate reports only; participant CSV not loaded.')

## Research design and leakage controls

- 625 participants in the fixed research split: 500 development and 125 test.
- Exactly 16 conceptual T0 inputs: baseline checkpoint ID plus 15 baseline questionnaire answers.
- Three targets are evaluated separately; overall concern is the logical OR of the three domain decisions.
- Participant-safe five-fold cross-validation repeated five times is used for the principal development comparisons.
- Preprocessing, calibration, and threshold selection are fitted inside development/training boundaries.
- T1 answers, follow-up scores, targets, participant identifiers, and future-derived values are prohibited from model inputs.
- The locked 125-person test partition was evaluated once for the original frozen release. Later TabICLv2 evaluation on that already-used cohort is secondary exploratory evidence, not fresh independent validation.

In [ ]:
techniques = pd.DataFrame([
    ('Logistic Regression', 'Core comparison', 'Linear, class-weighted L2 model'),
    ('RBF SVM', 'Core comparison', 'Non-linear kernel classifier'),
    ('Random Forest', 'Core comparison', 'Bagged decision trees'),
    ('Extra Trees', 'Core comparison', 'Extremely randomized trees'),
    ('XGBoost', 'Core comparison', 'Regularized gradient-boosted trees'),
    ('CatBoost', 'Additional shared-fold comparison', 'Ordered boosting with native categorical checkpoint'),
    ('Explainable Boosting Machine', 'Additional shared-fold comparison', 'Interpretable additive boosting'),
    ('LightGBM', 'Separate exploratory evaluation', 'Leaf-wise gradient boosting'),
    ('FIGS', 'Separate exploratory evaluation', 'Rule/tree-sum model'),
    ('Histogram Gradient Boosting', 'Separate exploratory evaluation', 'Histogram-based gradient boosting'),
    ('TabICLv2', 'Separate exploratory evaluation', 'Pretrained tabular in-context learner'),
], columns=['Algorithm family', 'Evaluation stage', 'Technique'])
display(techniques.style.hide(axis='index'))
print(f'Total individual algorithm families evaluated: {len(techniques)}')

## Assemble the development evidence

The following cell reads the already-generated aggregate reports. It normalizes their summary metrics into one presentation table without recomputing predictions.

In [ ]:
DOMAIN_LABELS = {
    'target_communication_concern': 'Communication',
    'target_cognitive_concern': 'Cognitive',
    'target_social_emotional_concern': 'Social-emotional',
}
NAME_LABELS = {
    'logistic_l2_balanced': 'Logistic Regression',
    'svm_rbf_balanced': 'RBF SVM',
    'random_forest_balanced_leaf5': 'Random Forest',
    'extra_trees_balanced_leaf5': 'Extra Trees',
    'xgboost_regularized_balanced': 'XGBoost',
    'catboost_ordered_balanced': 'CatBoost',
    'ebm_main_effects_balanced': 'Explainable Boosting Machine',
}
METRICS = ['average_precision_pr_auc', 'roc_auc', 'sensitivity', 'specificity', 'balanced_accuracy', 'brier_score']
rows = []

seven = load_report('development-additional-model-comparison-report.json')
for target, domain_data in seven['domains'].items():
    for candidate, candidate_data in domain_data['candidates'].items():
        summary = candidate_data['repeat_metric_summary']
        rows.append({'Domain': DOMAIN_LABELS[target], 'Algorithm': NAME_LABELS[candidate],
                     'Stage': 'Core' if candidate in list(NAME_LABELS)[:5] else 'Additional',
                     **{metric: summary[metric]['mean'] for metric in METRICS}})

separate_reports = [
    ('development-lightgbm-comparison-report.json', 'LightGBM'),
    ('development-figs-comparison-report.json', 'FIGS'),
    ('development-histgradientboosting-comparison-report.json', 'Histogram Gradient Boosting'),
    ('development-tabiclv2-model-comparison-report.json', 'TabICLv2'),
]
for filename, label in separate_reports:
    report = load_report(filename)
    for target, domain_data in report['domains'].items():
        summary = domain_data['repeat_metric_summary']
        rows.append({'Domain': DOMAIN_LABELS[target], 'Algorithm': label, 'Stage': 'Separate exploratory',
                     **{metric: summary[metric]['mean'] for metric in METRICS}})

development = pd.DataFrame(rows)
assert len(development) == 33
assert development['Algorithm'].nunique() == 11
display(development.sort_values(['Domain', 'average_precision_pr_auc'], ascending=[True, False]).style.hide(axis='index').format({m: '{:.3f}' for m in METRICS}))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 7), sharex=True)
for axis, domain in zip(axes, DOMAIN_LABELS.values()):
    view = development[development['Domain'] == domain].sort_values('average_precision_pr_auc')
    colors = view['Stage'].map({'Core': '#168ac2', 'Additional': '#ea579c', 'Separate exploratory': '#697985'})
    axis.barh(view['Algorithm'], view['average_precision_pr_auc'], color=colors)
    axis.set_title(domain)
    axis.set_xlabel('Mean development PR-AUC')
    axis.set_xlim(0, max(0.7, development['average_precision_pr_auc'].max() + 0.05))
    axis.grid(axis='x', alpha=.2)
fig.suptitle('Development evidence by domain (experiment stages shown together, not pooled)', fontsize=14)
fig.tight_layout()
plt.show()

### Reading the comparison

A higher PR-AUC indicates better ranking of the uncommon positive class. It is not a decision threshold and is not a probability of developmental concern. Differences across separately executed experiments should be interpreted with their uncertainty and protocol limitations, not as definitive head-to-head superiority.

In [ ]:
best_rows = []
for domain, group in development.groupby('Domain', sort=False):
    for metric in ['average_precision_pr_auc', 'roc_auc', 'balanced_accuracy', 'brier_score']:
        chosen = group.loc[group[metric].idxmin() if metric == 'brier_score' else group[metric].idxmax()]
        best_rows.append({'Domain': domain, 'Metric': metric, 'Numerically strongest algorithm': chosen['Algorithm'], 'Value': chosen[metric], 'Stage': chosen['Stage']})
display(pd.DataFrame(best_rows).style.hide(axis='index').format({'Value': '{:.3f}'}))

## Selected prototype architecture

The current local experimental prototype uses domain-specific models rather than forcing one algorithm across all outcomes:

- **Communication:** XGBoost
- **Cognitive:** TabICLv2
- **Social-emotional:** XGBoost
- **Overall decision:** logical OR of the three domain decisions

This is a research-prototype selection, not clinical approval. Cognitive TabICLv2 is locally integrated but is not a production-deployed or independently validated clinical model.

In [ ]:
locked = load_report('final-locked-test-report.json')
secondary_tabicl = load_report('secondary-tabiclv2-test-evaluation-report.json')
selected_test_rows = []
for target in ['target_communication_concern', 'target_social_emotional_concern']:
    item = locked['domains'][target]
    selected_test_rows.append({'Domain': DOMAIN_LABELS[target], 'Model': 'XGBoost', 'Evidence': 'Original one-time locked test', **item['metrics']})
cognitive = secondary_tabicl['domains']['target_cognitive_concern']['tabiclv2']
selected_test_rows.append({'Domain': 'Cognitive', 'Model': 'TabICLv2', 'Evidence': 'Secondary exploratory test', **cognitive['metrics']})
selected_test = pd.DataFrame(selected_test_rows)
display(selected_test[['Domain', 'Model', 'Evidence', 'average_precision_pr_auc', 'roc_auc', 'sensitivity', 'specificity', 'balanced_accuracy', 'brier_score']].style.hide(axis='index').format({m: '{:.3f}' for m in METRICS}))

### Critical evidence distinction

The Communication and Social-emotional values above come from the original one-time locked-test evaluation. The Cognitive TabICLv2 values come from a later evaluation on the **previously evaluated** 125-person cohort. That later result is explicitly secondary exploratory evidence: it is not fresh independent validation, external validation, confirmatory evidence, or clinical validation. These evidence categories must not be combined into a single claim of independently validated three-domain performance.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
for axis, row in zip(axes, selected_test_rows):
    cm = row['confusion_matrix']
    matrix = [[cm['true_negative'], cm['false_positive']], [cm['false_negative'], cm['true_positive']]]
    image = axis.imshow(matrix, cmap='Blues')
    for i in range(2):
        for j in range(2):
            axis.text(j, i, matrix[i][j], ha='center', va='center', fontsize=13)
    axis.set_xticks([0, 1], ['Predicted −', 'Predicted +'])
    axis.set_yticks([0, 1], ['Observed −', 'Observed +'])
    axis.set_title(f"{row['Domain']} — {row['Model']}")
fig.suptitle('Selected prototype models: test-cohort confusion matrices (different evidence status)')
fig.tight_layout()
plt.show()

## Why there is no training-loss curve

The experiments did not preserve a comparable per-iteration training/validation loss history across all models. XGBoost declares log loss as an evaluation metric and CatBoost declares Logloss as its optimization objective, but those declarations are not saved learning curves. Several candidates do not have a meaningful common epoch axis, and TabICLv2 uses a pretrained checkpoint rather than being trained from scratch in this project.

Therefore, this notebook does not invent or reconstruct a misleading training-loss graph. Generalisation is assessed using participant-safe repeated validation, held-out metrics, Brier score, calibration, and test evidence with clearly labelled status.

## Explainability methods used

| Model family | Research explanation evidence | Caregiver-facing implementation |
|---|---|---|
| Logistic Regression | Standardized coefficient direction and stability | Not used by current three-model prototype |
| Tree candidates | Held-out permutation importance for evaluation | XGBoost uses local model-native contributions |
| TabICLv2 | Bounded local response-variation analysis | Up to three relevant answer observations |

These are model-behaviour explanations, not causal explanations, clinical importance estimates, or diagnoses.

## Main scientific limitations

1. The cohort is limited in size and source; external generalisability has not been demonstrated.
2. Five repeated cross-validation estimates share participants, so their intervals are descriptive rather than five independent studies.
3. All eleven algorithms were not evaluated in one simultaneous experiment; later candidates were assessed in separately documented stages.
4. Thresholds and calibration were selected using development outcomes and may be selection-optimistic.
5. The TabICLv2 test-cohort evaluation is secondary exploratory evidence because that cohort had already been evaluated.
6. The targets are questionnaire-defined follow-up concerns, not diagnoses.
7. No model has fresh external, independent, prospective, or clinical validation.
8. Results must not be presented as replacing paediatric or developmental assessment.

## Viva conclusion

This research evaluated eleven tabular algorithm families across controlled and exploratory stages. The evidence supports a domain-specific prototype architecture—XGBoost for Communication and Social-emotional prediction, and experimental TabICLv2 for Cognitive prediction—because algorithm performance differs by outcome. The contribution is the leakage-controlled multi-domain evaluation, calibrated domain decisions, model-specific explanation boundary, and safety-constrained downstream activity retrieval. The system remains a non-diagnostic research prototype requiring fresh independent and external validation before any clinical or production claim.